# PRIMER PASO - Transformación de datos — demanda por tramo horario

Objetivo: construir, a partir de `informe_NOUP.csv`, un dataset limpio con el **target de
nº de citas por tramo horario y día** (mañana: antes de las 14:00, tarde: 14:00 en adelante),
sin desglosar por producto (se deja para una segunda fase).

Este notebook cubre la fase de **Transformación / Data Understanding**: deja el dataset
listo y ya dividido en train/test para que la fase de **EDA** (`eda.ipynb`) y la de
**Feature Engineering** (`feature_engineering.ipynb`) trabajen únicamente sobre el
histórico cerrado, sin tocar el conjunto de test.

**La lógica de cada paso vive en `src/utils/transform.py`**, no aquí — este notebook solo
llama a esas funciones y muestra el resultado de cada una. Así la misma transformación se
puede volver a aplicar en el futuro (p. ej. cuando llegue un nuevo export de reservas) sin
copiar y pegar código de un notebook. La única excepción es el split train/test (paso 10):
se queda directamente en el notebook, sin función propia — ver esa sección para el porqué.

Pasos:
1. Cargar el CSV crudo y descartar la fila de totales.
2. Parsear `Disponibilidad` en fecha de la cita + hora de inicio.
3. Quedarnos con reservas de servicio (excluir tarjetas de regalo / membresías) y deduplicar a nivel de reserva.
4. Quedarnos con las reservas confirmadas (no canceladas).
5. Aplicar un corte temporal fijo: solo citas ejecutadas hasta el 30/06/2026 (inclusive), para no meter en el modelo demanda todavía en curso o futura.
6. Asignar el tramo horario (mañana / tarde) según la hora de inicio.
7. Construir la rejilla completa fecha × tramo, agregar el nº de citas (target) y añadir las variables de calendario.
8. Comprobaciones de calidad.
9. Guardado del dataset completo.
10. División train / test **cronológica** (antes de la EDA, para evitar data leakage).

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append(str(Path('..').resolve()))
from src.utils.transform import (
    cargar_csv_bruto,
    parsear_fechas,
    filtrar_reservas_servicio,
    filtrar_confirmadas,
    aplicar_corte_temporal,
    asignar_tramo,
    construir_rejilla,
)

pd.set_option('display.max_columns', None)

RAW_PATH = Path('../informe_NOUP.csv')
OUTPUT_PATH = Path('../data/processed/ocupacion_tramos.csv')
TRAIN_PATH = Path('../data/processed/train.csv')
TEST_PATH = Path('../data/processed/test.csv')

# Solo se usan citas ejecutadas hasta esta fecha (inclusive): evita meter en el
# modelo demanda todavía en curso (reservas futuras o muy recientes al corte de datos)
FECHA_CORTE = pd.Timestamp('2026-06-30')

ModuleNotFoundError: No module named 'holidays'

## 1. Carga del CSV crudo

In [ ]:
df_raw = cargar_csv_bruto(RAW_PATH)

print(f"Filas: {len(df_raw)} | Reservas únicas: {df_raw['ID de reserva'].nunique()}")
df_raw.head(3)

## 2. Parseo de `Disponibilidad`: fecha y hora de la cita

`Disponibilidad` mezcla varios formatos: `"9/5/24 a las 12:00 – 13:40"` (rango),
`"9/5/24 a las 20:00"` (solo inicio) y `"3/5/24"` (sin hora — bonos, se descartan más
adelante). El parseo (`parse_disponibilidad`) vive en `src/transform.py`.

In [2]:
df_raw = parsear_fechas(df_raw)

print(f"Filas sin fecha_cita parseable: {df_raw['fecha_cita'].isna().sum()}")

NameError: name 'parsear_fechas' is not defined

## 3. Reservas de servicio, deduplicadas

Cada reserva puede tener varias filas (pago + reembolso, pago dividido...). Se excluyen
tarjetas de regalo / membresías (no son citas con horario) y se deduplica a nivel de
`ID de reserva`, quedándonos con la transacción más antigua de cada una.

In [ ]:
reservas = filtrar_reservas_servicio(df_raw)

print(f"Reservas de servicio (dedup): {len(reservas)}")
reservas['¿Cancelado?'].value_counts(dropna=False)

## 4. Solo reservas confirmadas (no canceladas)

El target debe reflejar demanda real atendida, así que se excluyen las reservas `Cancelled`
(~1,4% del total) y cualquier fila sin fecha/hora de cita parseada.

In [ ]:
antes = len(reservas[reservas['¿Cancelado?'] == 'No'])
confirmadas = filtrar_confirmadas(reservas)

print(f"Reservas confirmadas: {antes} | con cita válida (fecha + hora): {len(confirmadas)}")

## 5. Corte temporal: solo citas ejecutadas hasta el 30/06/2026 (inclusive)

Para no meter en el modelo demanda todavía en curso (reservas de fechas futuras, o de
fechas muy recientes cuyo recuento aún puede crecer porque siguen llegando reservas), se
descarta cualquier cita posterior a `FECHA_CORTE`. Todo lo que viene después (rejilla,
target, features) se construye solo con este histórico ya cerrado.

In [ ]:
antes = len(confirmadas)
confirmadas = aplicar_corte_temporal(confirmadas, FECHA_CORTE)

print(f"Reservas confirmadas antes del corte temporal: {antes}")
print(f"Reservas confirmadas hasta {FECHA_CORTE.date()} (inclusive): {len(confirmadas)}")
print(f"Descartadas por ser posteriores al corte: {antes - len(confirmadas)}")

## 6. Tramo horario (mañana / tarde, corte a las 14:00)

In [ ]:
confirmadas = asignar_tramo(confirmadas)

confirmadas.groupby('tramo').size()

## 7. Rejilla fecha × tramo, target y variables de calendario

`construir_rejilla` (en `src/transform.py`) construye el calendario completo desde la
primera cita hasta `FECHA_CORTE`, cruzado con los dos tramos, para que los días/tramos sin
ninguna reserva queden explícitos como `0` en vez de faltar en el dataset — y de paso
añade las variables de calendario (`dia_semana`, `nombre_dia`, `es_finde`, `mes`, `anio`,
`semana_iso`): son la descomposición directa de la fecha, no vienen de ningún análisis, así
que tiene sentido calcularlas aquí mismo, antes del split, para que la EDA las tenga
disponibles desde el principio. Las variables *derivadas con criterio* (a partir de lo que
muestre la EDA) van en `feature_engineering.ipynb`, no aquí.

In [ ]:
dataset = construir_rejilla(confirmadas, FECHA_CORTE)

assert dataset['n_citas'].sum() == len(confirmadas)
dataset.head()

## 8. Comprobaciones de calidad

In [ ]:
print("Rango de fechas:", dataset['fecha_cita'].min().date(), "->", dataset['fecha_cita'].max().date())
print("Filas totales (fecha x tramo):", len(dataset))
print("\nNulos por columna:")
print(dataset.isna().sum())

print("\nTotal de citas en el dataset transformado:", dataset['n_citas'].sum())
print("Total de reservas confirmadas de origen (hasta el corte):", len(confirmadas))

print("\nMedia de n_citas por tramo:")
print(dataset.groupby('tramo')['n_citas'].mean())

## 9. Guardado del dataset transformado

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
dataset.to_csv(OUTPUT_PATH, index=False)
print(f"Guardado en: {OUTPUT_PATH.resolve()}")
dataset.tail()

## 10. División train / test (split cronológico)

La guía del proyecto pide hacer el split train/test **antes** de la EDA, para no explorar
ni tomar decisiones contaminadas por el conjunto de test. Al ser una serie temporal, el
split **no puede ser aleatorio** (mezclar fechas al azar filtraría información del futuro
al pasado, algo que en producción nunca tendríamos) — se reserva como test el tramo
cronológico más reciente, respetando el orden temporal. Se mantiene el mismo `test_size`
de referencia que usa la guía (20%), aplicado aquí sobre el eje temporal.

Este paso se hace directamente aquí, sin extraerlo a una función en `src/` — llamarla
`split_train_test` se prestaba a confundirla con `train_test_split` de scikit-learn, que
hace un split **aleatorio** y no serviría para series temporales; mejor dejar la lógica
explícita en el notebook que arriesgarse a esa confusión.

A partir de aquí, **`eda.ipynb` y `feature_engineering.ipynb` solo deben leer `train.csv`**;
`test.csv` se reserva para la evaluación final del modelo.

In [ ]:
TEST_SIZE = 0.2  # mismo ratio que usa la guía para el split genérico, aplicado aquí de forma cronológica

fechas_unicas = dataset['fecha_cita'].drop_duplicates().sort_values().reset_index(drop=True)
n_test_dias = int(np.ceil(len(fechas_unicas) * TEST_SIZE))
fecha_split = fechas_unicas.iloc[-n_test_dias]

train = dataset[dataset['fecha_cita'] < fecha_split].copy()
test = dataset[dataset['fecha_cita'] >= fecha_split].copy()

print(f"Fecha de corte train/test: {fecha_split.date()}")
print(f"Train: {train['fecha_cita'].min().date()} -> {train['fecha_cita'].max().date()} "
      f"| {len(train)} filas ({train['fecha_cita'].nunique()} días)")
print(f"Test:  {test['fecha_cita'].min().date()} -> {test['fecha_cita'].max().date()} "
      f"| {len(test)} filas ({test['fecha_cita'].nunique()} días)")

train.to_csv(TRAIN_PATH, index=False)
test.to_csv(TEST_PATH, index=False)
print(f"\nGuardado train en: {TRAIN_PATH.resolve()}")
print(f"Guardado test en:  {TEST_PATH.resolve()}")

# SEGUNDO PASO - EDA dirigido al modelado — demanda por tramo horario

Fase 4 de la guía: EDA Dirigido al Modelado. Trabajamos solo con `data/processed/train.csv`
(generado en `transform.ipynb`) — test no se toca hasta la evaluación final, para no
contaminar las decisiones con el periodo que luego usaremos para validar.

Target: `n_citas`, nº de citas confirmadas por día y tramo (`mañana`/`tarde`, corte 14:00).

Pasos: primer vistazo → análisis del target → target vs. calendario → correlaciones y
outliers → un par de análisis extra → conclusiones para Feature Engineering.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')

TRAIN_PATH = Path('../data/processed/train.csv')
TARGET = 'n_citas'
DIAS_ORDEN = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

train = pd.read_csv(TRAIN_PATH, parse_dates=['fecha_cita'])
print(f"Shape: {train.shape}")
train.head()

## 1. Primer vistazo al train

Comprobación rápida de que el dataset está sano antes de analizarlo: tipos, nulos y
rangos de valores.

In [ ]:
train.info()
print("\nNulos por columna:")
print(train.isnull().mean() * 100)
train.describe(include='all')

## 2. Análisis del target

`n_citas` es un conteo (entero, ≥0), no una variable continua cualquiera — eso ya influye
en qué modelo y qué métrica tienen sentido más adelante. Miramos su distribución, si está
sesgada, y cuántos ceros tiene.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

train[TARGET].hist(bins=30, ax=axes[0])
axes[0].set_title('Distribución de n_citas (todo el train)')
axes[0].set_xlabel('n_citas')

for tramo, color in zip(['mañana', 'tarde'], ['#8a6bbf', '#4a2e8a']):
    sns.kdeplot(train.loc[train['tramo'] == tramo, TARGET], ax=axes[1], label=tramo, fill=True, alpha=0.3)
axes[1].set_title('Densidad de n_citas por tramo')
axes[1].legend()
plt.tight_layout()
plt.show()

print(f"Skewness global: {train[TARGET].skew():.2f}")
print("\nDescriptivos por tramo:")
print(train.groupby('tramo')[TARGET].describe())
print("\n% de días-tramo con 0 citas, por tramo:")
print((train.groupby('tramo')[TARGET].apply(lambda s: (s == 0).mean()) * 100).round(1))

**Lectura:** asimetría moderada (0,94) — hay días de mucha demanda pero son raros, no hace
falta transformar el target. Mañana y tarde son claramente dos escalas distintas (2,5 vs.
4,5 de media), así que a partir de aquí siempre analizamos por tramo por separado. Y ojo a
los ceros: el 21% de las mañanas no tienen ninguna cita, frente a solo el 5% de las tardes
— con esto el MAPE queda descartado como métrica (no se puede calcular sobre reales a 0),
usaremos MAE o RMSE.

## 3. Target vs. variables de calendario

Cómo se relaciona `n_citas` con cada variable — esto es lo que decide qué features
construir después. Siempre por tramo, nunca agregado.

### 3.1 Tramo

In [ ]:
# tramo
plt.figure(figsize=(5, 4))
sns.boxplot(data=train, x='tramo', y=TARGET)
plt.title('n_citas por tramo')
plt.show()

print(train.groupby('tramo')[TARGET].mean().round(2))

**Lectura:** la tarde casi duplica a la mañana (4,54 vs. 2,52) — es la variable individual
con más peso, el modelo la necesita sí o sí.

### 3.2 Día de la semana

¿La demanda sube poco a poco a lo largo de la semana, o hay saltos?

In [ ]:
# día de la semana (por tramo, para no mezclar escalas)
plt.figure(figsize=(9, 4))
sns.boxplot(data=train, x='nombre_dia', y=TARGET, hue='tramo', order=DIAS_ORDEN)
plt.title('n_citas por día de la semana y tramo')
plt.xticks(rotation=30)
plt.show()

tabla_dow = train.groupby(['nombre_dia', 'tramo'])[TARGET].mean().unstack().reindex(DIAS_ORDEN)
tabla_dow

**Lectura:** hay saltos, no gradiente. De lunes a jueves la demanda está plana y baja. El
**viernes** despega, pero solo por la tarde (6,1). El **sábado** es el pico absoluto (4,7
mañana / 6,9 tarde), y el **domingo** destaca sobre todo por la mañana (4,1, más del doble
que cualquier mañana entre semana). Con esto vale más una categoría (entre semana /
viernes / fin de semana) que el número de día tal cual — es el origen de `grupo_dia`.

### 3.3 Mes (estacionalidad anual)

¿Hay meses sistemáticamente más fuertes que otros?

In [ ]:
# mes (estacionalidad anual, por tramo)
tabla_mes = train.groupby(['mes', 'tramo'])[TARGET].mean().unstack()
tabla_mes.plot(kind='bar', figsize=(9, 4), color=['#8a6bbf', '#4a2e8a'])
plt.title('Media de n_citas por mes y tramo')
plt.ylabel('n_citas (media)')
plt.xticks(rotation=0)
plt.show()

tabla_mes

**Lectura:** sí, y con lógica de negocio clara — pico en **invierno** (febrero, con
diferencia), valle en **verano**. Un spa es plan de frío, y en Sevilla el verano compite
con playa y vacaciones. Un detalle a cuidar: diciembre y enero se comportan parecido pero
son extremos opuestos en la escala numérica del mes (12 y 1) — mejor agrupar por
`temporada` que dejar que un modelo trate el mes como un número lineal. Por lo que puede interesar hacer una variable de trimestre que recoja las estaciones.

### 3.4 Fin de semana vs. entre semana

Versión resumen del punto anterior: ¿cuánto vale el fin de semana en bloque?

In [ ]:
# fin de semana vs. entre semana
tabla_finde = train.groupby(['es_finde', 'tramo'])[TARGET].mean().unstack()
tabla_finde.index = tabla_finde.index.map({True: 'Fin de semana', False: 'Entre semana'})
tabla_finde.plot(kind='bar', figsize=(5, 4), color=['#8a6bbf', '#4a2e8a'])
plt.title('Media de n_citas: fin de semana vs. entre semana')
plt.xticks(rotation=0)
plt.show()

tabla_finde

**Lectura:** el fin de semana multiplica la mañana por 2,5 y sube la tarde un 57%. Pero
esta variable esconde al viernes, que ya vimos que se comporta como fin de semana solo por
la tarde — por eso `grupo_dia` (con el viernes aparte) sustituirá a `es_finde`.

### 3.5 Tendencia de fondo

Falta la tercera pieza de cualquier serie temporal: ¿el negocio crece con el tiempo?
Agregamos por semana para ver la dirección sin el ruido diario.

In [ ]:
# tendencia de fondo: evolución semanal por tramo
semanal = (
    train.assign(semana=train['fecha_cita'] - pd.to_timedelta(train['dia_semana'], unit='D'))
         .groupby(['semana', 'tramo'])[TARGET].sum()
         .unstack()
)
semanal.plot(figsize=(11, 4), color=['#8a6bbf', '#4a2e8a'])
plt.title('Reservas por semana y tramo (train)')
plt.ylabel('n_citas (suma semanal)')
plt.show()

**Lectura:** crecimiento claro, sobre todo en 2024 (de ~10-20 citas semanales al abrir a
60-90 un año después); a partir de mediados de 2025 se estabiliza más, con los picos de
invierno por encima. Como esto es tendencia y no calendario, hace falta una variable
aparte (`dias_desde_inicio`) — las variables cíclicas no distinguen un año de otro.

## 4. Correlaciones y outliers

### 4.1 Correlaciones

Convertimos `es_finde` y `tramo` a 0/1 para poder incluirlas. Aviso: Pearson solo mide
relación lineal — para algo cíclico como `mes` es casi ciego (diciembre y enero son
vecinos en la realidad, opuestos en la escala), así que una correlación baja ahí no
significa que el mes no importe.

In [ ]:
corr_df = train.copy()
corr_df['es_finde'] = corr_df['es_finde'].astype(int)
corr_df['tramo_tarde'] = (corr_df['tramo'] == 'tarde').astype(int)

corr = corr_df[['n_citas', 'dia_semana', 'es_finde', 'mes', 'anio', 'semana_iso', 'tramo_tarde']].corr()

plt.figure(figsize=(7, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlación con n_citas (train)')
plt.show()

**Lectura:** lo más correlacionado con el target es `dia_semana` (0,40), `es_finde` (0,39),
`tramo_tarde` (0,36) y `anio` (0,31) — coincide con todo lo visto. `mes` y `semana_iso`
salen casi a cero, pero es la ceguera de Pearson ante ciclos, no que no importen (la
sección 3.3 ya lo demostró). `dia_semana` y `es_finde` están muy correlacionadas entre sí
(una deriva de la otra) — redundancia que resolveremos quedándonos con `grupo_dia`.

### 4.2 Outliers
Lo interesante no es cuántos hay, sino si son errores o demanda real.

In [ ]:
# outliers por IQR, calculados por separado en cada tramo (escalas distintas)
def outliers_iqr(serie):
    q1, q3 = serie.quantile(0.25), serie.quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return (serie < lo) | (serie > hi)

for tramo in ['mañana', 'tarde']:
    sub = train.loc[train['tramo'] == tramo, TARGET]
    mask = outliers_iqr(sub)
    print(f"Tramo {tramo}: {mask.sum()} outliers de {len(sub)} ({mask.mean()*100:.1f}%)")

train.loc[
    train.groupby('tramo')[TARGET].transform(outliers_iqr)
].sort_values(TARGET, ascending=False).head(16)

**Lectura:** muy pocos (1,3% por tramo) y todos por arriba. Al mirar las fechas, son
sábados y días de febrero (San Valentín, sábados alrededor) — demanda real en fechas pico,
no errores. Se mantienen todos: son justo lo que el modelo tiene que aprender a predecir.

## 5. Tres preguntas más

### 5.1 Día de la semana × mes

¿El sábado pega igual de fuerte en febrero que en julio, o cambia con la temporada?

In [ ]:
diario = train.groupby('fecha_cita').agg(
    n_citas=('n_citas', 'sum'),
    dia_semana=('dia_semana', 'first'),
    mes=('mes', 'first'),
)

pivot = diario.pivot_table(index='dia_semana', columns='mes', values='n_citas', aggfunc='mean')
pivot.index = [DIAS_ORDEN[i] for i in pivot.index]

plt.figure(figsize=(11, 4.5))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='Purples', cbar_kws={'label': 'citas/día (media)'})
plt.title('Media de citas diarias (mañana + tarde) por día de la semana y mes')
plt.xlabel('mes')
plt.ylabel('')
plt.show()

**Lectura:** cambia mucho — los efectos se amplifican entre sí. El sábado de febrero llega
a 20,8 citas/día de media; el sábado de julio se queda en 7,6. En verano la semana entera
se aplana (sábado apenas duplica a un martes); en febrero el sábado casi lo quintuplica.
Los modelos de árboles capturan esto solos, sin que haga falta crear la interacción a
mano — otro punto a favor de árboles/gradient boosting para este problema.

### 5.2 Crecimiento interanual: ¿estación o tendencia?

Para separar ambas cosas, comparamos el mismo mes en años distintos.

In [ ]:
mensual = train.groupby(['anio', 'mes'])['n_citas'].sum().unstack('anio')

mensual.plot(figsize=(9, 4), marker='o')
plt.title('Total de citas por mes, una línea por año (train)')
plt.ylabel('n_citas (total mensual)')
plt.xticks(range(1, 13))
plt.show()

mensual

**Lectura:** es crecimiento real, no solo estación — mayo pasa de 52 citas (2024) a 293
(2025), ×5,6. La brecha entre años se estrecha con el tiempo, coherente con el
"aplanamiento" que vimos en 3.5. Aviso: 2024 solo tiene datos desde mayo (apertura), y el
único punto de 2026 (enero) es un mes incompleto por el corte del split — no leerlo como
caída.

### 5.3 Días sin ninguna cita: ¿demanda cero o cierre?

No es lo mismo "no vino nadie" que "estaba cerrado". Miramos los días con cero citas en
ambos tramos a la vez.

In [ ]:
total_diario = train.groupby('fecha_cita')['n_citas'].sum()
dias_cero = total_diario[total_diario == 0]

print(f"Días con cero citas en ambos tramos: {len(dias_cero)} de {len(total_diario)} "
      f"({len(dias_cero) / len(total_diario) * 100:.1f}%)\n")
for fecha in dias_cero.index:
    print(fecha.strftime('%Y-%m-%d  %A'))

**Lectura:** 23 días (3,7%), y la mayoría son cierres, no demanda cero: Navidad, Año Nuevo
y Reyes se repiten los dos años, y hay dos bloques seguidos (9-13 marzo y 17-19 noviembre
de 2025) que tienen toda la pinta de vacaciones u obras. El resto son días sueltos de
2024, cuando el negocio acababa de abrir.

Esto explica algo importante: `es_festivo` salió muy débil en Feature Engineering porque
mezcla dos efectos opuestos que se cancelan (festivos que cierran el spa vs. festivos que
traen más clientes). Y para Preprocesado: esos cierres no son predecibles desde el
calendario público, conviene marcarlos o excluirlos de train.

## 6. Conclusiones

**Para Feature Engineering:**

- `tramo` manda en la escala (tarde casi dobla a mañana) → `tramo_tarde` numérica.
- El patrón semanal da saltos, no gradiente (plano L-J, sube viernes tarde, pico fin de
  semana) → `grupo_dia` en vez de `dia_semana`/`es_finde` a solas.
- Estacionalidad anual clara (invierno arriba, verano abajo, diciembre-enero pegados) →
  `temporada` en vez de confiar en `mes` como número lineal.
- Tendencia de crecimiento real (mayo ×5,6 de 2024 a 2025) → variable de tendencia
  (`dias_desde_inicio`), las cíclicas no la capturan.
- Dataset limpio, outliers pocos y explicables como demanda real en fechas pico → no se
  toca nada.

**Avisos para más adelante:**

- MAPE inviable (21% de mañanas a 0) → MAE o RMSE.
- El efecto fin de semana se dispara en temporada alta — favorece modelos de árboles sobre
  lineales.
- Hay cierres reales del negocio (Navidad/Reyes + dos bloques en 2025) mezclados con
  demanda cero — por eso `es_festivo` sale débil, y en Preprocesado convendría marcarlos.

# TERCER PASO - Feature Engineering — demanda por tramo horario

Fase 5 de la guía. Partimos de las conclusiones de `eda.ipynb` y aplicamos siempre la
misma transformación a train y a test.

**La lógica de cada variable vive en `src/feature_engineering.py`**, no aquí — este
notebook solo llama a esas funciones y verifica el resultado. Es la misma función que se
usará para predecir una fecha real en el futuro (`construir_features(fecha, tramo)`), así
que entrenamiento y producción calculan las variables exactamente igual.

Requiere el paquete `holidays` (`pip install -r requirements.txt`).

Variables:
1. `trimestre` y `dias_desde_inicio` (tendencia).
2. `grupo_dia` y `temporada` (saltos semanales y estación).
3. `tramo_tarde` (versión 0/1 de `tramo`).
4. `es_festivo` (calendario oficial de Andalucía).
5. `es_vispera_festivo` y `es_fecha_comercial` (vísperas y fechas señaladas tipo San
   Valentín).
6. `es_cierre` — **marcador, no feature**: días en que el spa no abre (se explica en su
   sección).

Se evaluaron también *lags* (citas de fechas anteriores) y por ahora quedan fuera — el
porqué está en la sección 7.

## 0. Qué va aquí y qué venía ya de `transform.ipynb`

Las columnas de calendario del dataset (`dia_semana`, `mes`, `es_finde`...) son la
descomposición directa de la fecha: se crearon en `transform.ipynb`, antes del split, para
que la EDA pudiera agrupar por ellas desde el principio. Las de este notebook nacen de lo
que la EDA encontró (o de fuentes externas, como los festivos) — por eso llegan después.

Es normal que alguna variable nueva deje obsoleta a una de calendario — pasa con
`es_finde` → `grupo_dia` (sección 2). Y como estas variables no existían durante la EDA,
la sección 8 comprueba su relación con el target antes de guardarlas.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append(str(Path('..').resolve()))
from src.utils.feature_engineering import (
    anadir_tendencia,
    anadir_variables_negocio,
    anadir_tramo_tarde,
    anadir_festivos,
    anadir_vispera_y_comercial,
    anadir_cierre,
    primer_domingo_mayo,
    FESTIVOS_ANDALUCIA,
)

pd.set_option('display.max_columns', None)

TRAIN_PATH = Path('../data/processed/train.csv')
TEST_PATH = Path('../data/processed/test.csv')
TRAIN_OUT = Path('../data/processed/train_features.csv')
TEST_OUT = Path('../data/processed/test_features.csv')
TARGET = 'n_citas'

train = pd.read_csv(TRAIN_PATH, parse_dates=['fecha_cita'])
test = pd.read_csv(TEST_PATH, parse_dates=['fecha_cita'])

print(f"Train: {train.shape} | {train['fecha_cita'].min().date()} -> {train['fecha_cita'].max().date()}")
print(f"Test:  {test.shape} | {test['fecha_cita'].min().date()} -> {test['fecha_cita'].max().date()}")

## 1. Trimestre y tendencia

`trimestre` sale directo de la fecha.

`dias_desde_inicio` es la variable de tendencia que pedía la EDA (§3.5): las variables de
calendario se repiten cada año, así que para ellas mayo de 2024 y mayo de 2025 son
idénticos — cuando en realidad hubo 52 citas frente a 293. Este contador (días desde la
primera fecha de train) le da al modelo el eje de "cuánto ha crecido el negocio". La fecha
de referencia se fija con train y se reutiliza en test.

*(Se descartó la codificación seno/coseno de mes y día que hubo en una versión anterior:
complejidad extra que los modelos de árboles no necesitan.)*

*Aviso para Modelado: los árboles no extrapolan esta variable más allá del rango visto en
train — al predecir lejos se quedan en el nivel del final del histórico. Un modelo lineal
sí proyecta la tendencia.*

In [ ]:
FECHA_REFERENCIA = train['fecha_cita'].min()  # "aprendida" en train, reutilizada en test

train = anadir_tendencia(train, FECHA_REFERENCIA)
test = anadir_tendencia(test, FECHA_REFERENCIA)

train[['fecha_cita', 'trimestre', 'dias_desde_inicio']].head()

## 2. `grupo_dia` y `temporada`

Las dos salen directas de la EDA. La demanda semanal va a saltos, no en gradiente (plana
de lunes a jueves, sube el viernes por la tarde, pico el fin de semana, §3.2) →
`grupo_dia` con tres categorías. Y la estación pesa más que el número del mes — diciembre
y enero se comportan igual pero son 12 y 1 en la escala (§3.3) → `temporada`.

`grupo_dia` deja obsoleta a `es_finde`: es la misma información pero sin perder al
viernes. Se mantienen ambas de momento y `es_finde` se elimina en Preprocesado. Con `mes`
y `temporada` no pasa lo mismo: `mes` tiene el detalle fino y `temporada` es su agrupación
de negocio — se quedan las dos.

In [ ]:
train = anadir_variables_negocio(train)
test = anadir_variables_negocio(test)

train.groupby('grupo_dia')[TARGET].mean().round(2)

## 3. `tramo_tarde`

La variable con más peso según la EDA (§3.1: la tarde casi dobla a la mañana), en versión
0/1 para los modelos que no aceptan texto. Contiene exactamente la misma información que
`tramo` — aquí se conservan las dos por legibilidad, y `tramo` se elimina en Preprocesado.

In [ ]:
train = anadir_tramo_tarde(train)
test = anadir_tramo_tarde(test)

train[['tramo', 'tramo_tarde']].drop_duplicates()

## 4. `es_festivo`

Usamos la librería `holidays` con el calendario oficial de Andalucía (`subdiv='AN'`):
trae los festivos nacionales, el Día de Andalucía, Jueves y Viernes Santo en su fecha
correcta cada año, y los traslados a lunes cuando un festivo cae en domingo. En
`src/feature_engineering.py` se expande automáticamente a cualquier año que se consulte —
no hace falta fijar un rango de años como aquí, eso es importante para cuando esta misma
función se use para predecir una fecha real futura.

Lo que ninguna librería trae son las 2 fiestas locales de Sevilla capital (las decide el
Ayuntamiento cada año) — `FESTIVOS_LOCALES_SEVILLA` (en `src/feature_engineering.py`)
queda vacía, lista para rellenar consultando el BOJA.

Ojo con lo que cabe esperar de esta variable: la EDA (§5.3) encontró que el spa **cierra**
en algunos festivos (Navidad, Año Nuevo, Reyes), así que "festivo" mezcla días de cierre
con días de posible demanda extra — no va a ser una señal limpia.

In [ ]:
import holidays

anios = range(train['fecha_cita'].dt.year.min(), test['fecha_cita'].dt.year.max() + 1)

train = anadir_festivos(train)
test = anadir_festivos(test)

# Objeto aparte, solo para listar y verificar los festivos calculados en el rango de datos
# (la columna es_festivo en sí se calcula con FESTIVOS_ANDALUCIA, que se expande a cualquier año)
festivos_verificacion = holidays.Spain(subdiv='AN', years=list(anios))
print(f"Festivos calculados para {list(anios)}: {len(festivos_verificacion)}")
for fecha, nombre in sorted(festivos_verificacion.items()):
    print(fecha, '-', nombre)

## 5. `es_vispera_festivo` y `es_fecha_comercial`

Dos flags más, calculables para cualquier fecha futura solo con el calendario:

- `es_vispera_festivo`: día anterior a festivo. Plausible que la gente reserve spa cuando
  el día siguiente es libre. Sale del mismo calendario de la sección 4.
- `es_fecha_comercial`: fechas señaladas de regalo/pareja — el pico más alto de todo el
  histórico es el 14 de febrero (San Valentín), y en los datos originales existe incluso
  un producto "Ritual Día de la Madre". Incluimos San Valentín (fijo) y el Día de la Madre
  (primer domingo de mayo en España, se calcula solo).

Las dos se comprueban contra el target en la sección 8 antes de darlas por buenas.

In [ ]:
train = anadir_vispera_y_comercial(train)
test = anadir_vispera_y_comercial(test)

# Solo para el print de verificación (las fechas comerciales reales se calculan dentro de la función)
fechas_comerciales_verificacion = set()
for anio in anios:
    fechas_comerciales_verificacion.add(pd.Timestamp(year=anio, month=2, day=14))
    fechas_comerciales_verificacion.add(primer_domingo_mayo(anio))

print("Fechas comerciales:", sorted(d.date() for d in fechas_comerciales_verificacion))
print(f"Vísperas de festivo en train: {train['es_vispera_festivo'].sum() // 2} días")

### 5.1 ¿Hay más fechas comerciales que aporten señal?

Antes de dar `es_fecha_comercial` por cerrada, probamos otras fechas candidatas con
significado de regalo/consumo en España (Día del Padre, Día de la Mujer, Black Friday,
Cyber Monday, Nochevieja, Noche de San Juan) para ver si alguna muestra el mismo tipo de
repunte que San Valentín o el Día de la Madre — y si añadirlas reforzaría la señal o la
diluiría.

In [ ]:
def ultimo_viernes_noviembre(anio):
    fin_nov = pd.Timestamp(year=anio, month=11, day=30)
    return fin_nov - pd.Timedelta(days=(fin_nov.weekday() - 4) % 7)


candidatas = {
    'San Valentín (14 feb) — ya incluida': [pd.Timestamp(a, 2, 14) for a in anios],
    'Día de la Madre (1er dom mayo) — ya incluida': [primer_domingo_mayo(a) for a in anios],
    'Día del Padre (19 marzo)': [pd.Timestamp(a, 3, 19) for a in anios],
    'Día de la Mujer (8 marzo)': [pd.Timestamp(a, 3, 8) for a in anios],
    'Black Friday (últ. viernes nov)': [ultimo_viernes_noviembre(a) for a in anios],
    'Cyber Monday (lunes tras BF)': [ultimo_viernes_noviembre(a) + pd.Timedelta(days=3) for a in anios],
    'Nochevieja (31 dic)': [pd.Timestamp(a, 12, 31) for a in anios],
    'Noche de San Juan (23 jun)': [pd.Timestamp(a, 6, 23) for a in anios],
}

fmin, fmax = train['fecha_cita'].min(), train['fecha_cita'].max()
base = train.groupby('tramo')[TARGET].mean().round(2)
print(f"Media global de referencia -> mañana: {base['mañana']} | tarde: {base['tarde']}\n")

for nombre, fechas in candidatas.items():
    en_train = sorted(f for f in fechas if fmin <= f <= fmax)
    if not en_train:
        print(f"{nombre:45s} sin fechas dentro del rango de train")
        continue
    medias = train[train['fecha_cita'].isin(en_train)].groupby('tramo')[TARGET].mean().round(2)
    print(f"{nombre:45s} n={len(en_train)}  mañana={medias.get('mañana', float('nan')):>5}  "
          f"tarde={medias.get('tarde', float('nan')):>5}")

**Lectura — ninguna candidata nueva se incorpora, y además destapa un problema de fondo:**

- **La mayoría no muestra repunte.** Black Friday, Cyber Monday, Nochevieja y Noche de San
  Juan quedan igual o por debajo de la media global; Día del Padre se queda justo en la
  media.
- **Día de la Mujer es la excepción llamativa, y por eso mismo sospechosa**: mañana=11
  frente a una media de 2,52 (×4), pero tarde=5, prácticamente en la media (4,54). Un
  repunte real de "fecha de regalo" debería notarse en ambos tramos, como pasa con San
  Valentín — que suba solo uno de los dos con un único dato detrás huele más a casualidad
  de un día concreto que a patrón. No se incorpora.
- **El hallazgo de fondo es otro: `es_fecha_comercial` tiene un tamaño de muestra
  minúsculo.** Train solo cubre un San Valentín (14/02/2025) y un Día de la Madre
  (04/05/2025) — los demás años caen fuera del rango de train. La "señal fuerte" de la
  sección 8 está sostenida por **2 días sueltos (4 filas)**, y el 14/02/2025 es
  literalmente el outlier de tarde más alto de todo el dataset (15 citas, EDA §4.2). No es
  que la variable esté mal planteada — Valentín y el Día de la Madre son fechas de regalo
  reales, una con producto propio en el catálogo — pero con un solo caso por fecha no se
  distingue "patrón real" de "casualidad de un día muy bueno". Se mantiene, con esta
  reserva explícita, hasta que el histórico crezca y haya más de un año por fecha para
  confirmarlo.

### 5.2 Validación con el histórico completo (de cara al modelo final)

La reserva de la §5.1 era de tamaño de muestra: solo 1 ocurrencia de cada fecha dentro de
train. Pero el producto final no se queda con este split — cuando se entrene el modelo
definitivo, **todo el histórico actual (train + test) pasará a ser el nuevo train**, y lo
que hoy es test se sustituirá por fechas realmente futuras a predecir. Así que tiene
sentido comprobar cómo se ven estos patrones con **train + test juntos**, simulando esa
situación futura.

**Importante:** esto es solo una validación de cara a esa retrain futura, no cambia nada
del pipeline actual — `train_features.csv`/`test_features.csv` se generan igual que
siempre, sin tocar test. Es una comprobación aparte, con su propia carga de datos.

In [ ]:
# Carga aparte, solo para esta validación — no sustituye a train/test del pipeline
_completo_validacion = pd.concat(
    [pd.read_csv(TRAIN_PATH, parse_dates=['fecha_cita']), pd.read_csv(TEST_PATH, parse_dates=['fecha_cita'])],
    ignore_index=True,
)
_fmin, _fmax = _completo_validacion['fecha_cita'].min(), _completo_validacion['fecha_cita'].max()
_base = _completo_validacion.groupby('tramo')[TARGET].mean().round(2)

print(f"Histórico completo: {_fmin.date()} -> {_fmax.date()} ({len(_completo_validacion)} filas)")
print(f"Media global -> mañana: {_base['mañana']} | tarde: {_base['tarde']}\n")

for nombre, fechas in candidatas.items():
    en_rango = sorted(f for f in fechas if _fmin <= f <= _fmax)
    if not en_rango:
        continue
    sub = _completo_validacion[_completo_validacion['fecha_cita'].isin(en_rango)]
    medias = sub.groupby('tramo')[TARGET].mean().round(2)
    fechas_str = ', '.join(f.strftime('%Y-%m-%d') for f in en_rango)
    print(f"{nombre:35s} n={len(en_rango)}  mañana={medias.get('mañana', float('nan')):>5}  "
          f"tarde={medias.get('tarde', float('nan')):>5}   [{fechas_str}]")

**Lectura — con un año más de datos, el panorama cambia para las dos fechas buenas:**

- **San Valentín queda confirmado, no era casualidad.** Ahora hay 2 años: 2025 (7/15) y
  **2026 (10/20) — todavía más alto**. Las dos veces muy por encima de la media global
  (2,81/4,91), en ambos tramos, y subiendo. Es el candidato más sólido de todos.
- **Día de la Madre también se confirma y crece**: 2025 (4/6) y 2026 (9/10), las dos veces
  por encima de la media y con la misma dirección. Con dos años consistentes, deja de ser
  "un dato suelto".
- **El resto sigue sin mostrar patrón, ahora con más evidencia todavía de que es ruido**:
  Día del Padre se mantiene pegado a la media los dos años; **Día de la Mujer se
  contradice entre años** (2025 dispara mañana pero no tarde, 2026 al revés — justo el
  comportamiento errático que hacía sospechar en §5.1); Black Friday, Cyber Monday,
  Nochevieja y Noche de San Juan (con 3 años ya) siguen sin despegar de la media o por
  debajo.

**Conclusión para el modelo final**: cuando se reentrene con todo el histórico, San
Valentín y Día de la Madre pasan de "señal con reserva" a **señal confirmada con 2 años
consecutivos** — no haría falta ningún cambio en `es_fecha_comercial`, las fechas ya
elegidas eran las correctas. El resto de candidatas se descartan con más confianza
todavía que en train solo.

## 6. `es_cierre` — marcador, no feature del modelo

La EDA (§5.3) encontró que el spa no abre en Navidad, Año Nuevo y Reyes (cero citas ambos
años). Marcamos esos días, pero **esta columna no debe entrar al modelo como feature**: si
se sabe que está cerrado, no tiene sentido pedirle al modelo que estime las citas — la
respuesta es 0 por regla de negocio, no por predicción.

Su utilidad es otra:

- **En Preprocesado**: excluir estas filas de train. Si se quedan, el modelo aprende "los
  25 de diciembre la demanda es bajísima" cuando en realidad es que no se abrió, y eso
  contamina lo que aprende de los días normales de invierno.
- **En producción**: regla previa al modelo — si la fecha está marcada como cierre, se
  devuelve 0 y no se llama al modelo.

Solo marcamos las 3 fechas **confirmadas por el EDA** (se repiten los dos años),
`CIERRES_RECURRENTES` en `src/feature_engineering.py`. Los bloques de marzo y noviembre de
2025 que vimos también a cero NO se marcan porque no sabemos si fueron cierres o
casualidad — `CIERRES_CONOCIDOS` (mismo fichero) queda como lista manual para añadirlos si
el negocio lo confirma, igual que hicimos con los festivos locales.

In [ ]:
train = anadir_cierre(train)
test = anadir_cierre(test)

print(f"Días marcados como cierre en train: {train['es_cierre'].sum() // 2}")
print(f"Días marcados como cierre en test:  {test['es_cierre'].sum() // 2}")

## 7. Por qué no incluimos *lags* (de momento)

Un lag ("cuántas citas hubo hace X días en esta franja") predice bien — probamos 7, 14 y
364 días y la correlación con el target rondaba 0,55 — pero lo dejamos fuera por ahora:

- **Solo se puede calcular si la fecha a la que apunta ya ha pasado.** `lag_7` sirve para
  predecir hasta 7 días vista; este proyecto necesita también horizontes largos.
- **Un lag largo valdría para horizontes largos, pero se come el train**: con `lag_364` el
  58% de las filas se quedaba sin valor (no hay "hace un año" para el primer año del
  histórico).
- **En producción exige tener el histórico de reservas consultable automáticamente** cada
  vez que se predice — algo que aún no está decidido.

No es un descarte definitivo: cuando se concrete el horizonte de cada uso del modelo y la
disponibilidad del dato, se retoma. Mientras, la tendencia y la estacionalidad quedan
cubiertas por `dias_desde_inicio`, `mes`, `temporada` y `grupo_dia`, que se calculan para
cualquier fecha futura sin depender de nada.

## 8. Verificación contra el target

Estas variables se crearon después de la EDA, así que su relación con `n_citas` no se ha
comprobado aún. Chequeo rápido antes de guardarlas — si alguna no mostrara relación
ninguna, habría que revisarla en vez de pasarla al modelo sin más. Para `es_cierre` la
comprobación es distinta: debe salir con media ≈ 0, confirmando que marca bien los días
cerrados.

In [ ]:
print("Media de n_citas: festivo vs. no festivo (por tramo)")
print(train.groupby(['tramo', 'es_festivo'])[TARGET].mean().round(2))

print("\nMedia de n_citas por temporada (por tramo)")
print(train.groupby(['tramo', 'temporada'])[TARGET].mean().round(2))

print("\nMedia de n_citas: víspera de festivo vs. resto (por tramo)")
print(train.groupby(['tramo', 'es_vispera_festivo'])[TARGET].mean().round(2))

print("\nMedia de n_citas: fecha comercial vs. resto (por tramo)")
print(train.groupby(['tramo', 'es_fecha_comercial'])[TARGET].mean().round(2))

print("\nMedia de n_citas en días marcados como cierre (debe ser ~0)")
print(train.groupby('es_cierre')[TARGET].mean().round(2))

print("\nCorrelación con n_citas de las variables numéricas nuevas")
print(train[['dias_desde_inicio', TARGET]].corr()[TARGET].round(3))

**Lectura:**

- `es_fecha_comercial`: la señal nueva más fuerte con diferencia — esas fechas duplican
  con creces la media (mañana 5,5 vs. 2,5; tarde 10,5 vs. 4,5). Se queda, **pero con
  reserva**: son solo 2 fechas con 1 ocurrencia cada una en train (ver §5.1) — la señal es
  real en el sentido de que esos días pasaron, pero el tamaño de muestra es demasiado
  pequeño para hablar de patrón confirmado todavía.
- `temporada`: patrón claro en ambos tramos (invierno > otoño/primavera > verano). Útil.
- `dias_desde_inicio` (0,28): señal moderada, confirma la tendencia.
- `es_cierre`: media exactamente 0,00 en los días marcados — el marcador funciona.
- `es_festivo`: casi no mueve la media — como anticipaba la EDA (§5.3), los festivos de
  cierre y los de demanda extra se cancelan. Se mantiene pero es candidata a descartar.
- `es_vispera_festivo`: no muestra señal (la mañana incluso baja un poco). La hipótesis
  "reservan la víspera" no se cumple en estos datos. Se guarda igualmente, pero junto a
  `es_festivo` es la otra candidata clara a eliminar en Preprocesado si se simplifica.

## 9. Guardado

Notas para Preprocesado, todas juntas para no perder ninguna:

**Columnas redundantes a eliminar:**

| Eliminar | En favor de | Motivo |
|---|---|---|
| `tramo` | `tramo_tarde` | misma información, texto vs. 0/1 |
| `nombre_dia` | `dia_semana` | misma información, texto vs. número |
| `es_finde` | `grupo_dia` | contenida en `grupo_dia`, no distingue el viernes |

(`mes`/`temporada` y `dia_semana`/`grupo_dia` no son duplicados: detalle fino +
agrupación de negocio, se quedan ambos pares.)

**Tratamiento especial:**

- `es_cierre`: **no pasar al modelo como feature**. Excluir sus filas de train, y en
  producción devolver 0 por regla sin llamar al modelo (sección 6).
- `es_festivo` y `es_vispera_festivo`: señal débil verificada — primeras candidatas a
  eliminar si hay que simplificar.

In [ ]:
TRAIN_OUT.parent.mkdir(parents=True, exist_ok=True)
train.to_csv(TRAIN_OUT, index=False)
test.to_csv(TEST_OUT, index=False)

print(f"Guardado train con features en: {TRAIN_OUT.resolve()}")
print(f"Guardado test con features en:  {TEST_OUT.resolve()}")
print(f"\nColumnas finales ({train.shape[1]}): {list(train.columns)}")
train.tail()

## 10. Cómo se usará esto para predecir una fecha real

Todo lo de este notebook son llamadas a funciones de `src/feature_engineering.py` sobre
todo el histórico a la vez. Para predecir una fecha nueva en producción, la misma lógica
se usa con `construir_features(fecha, tramo)` — una fecha, un tramo, una fila. **`tramo`
tiene que darse como input, no se puede derivar de la fecha** (no hay forma de saber si se
pregunta por la mañana o la tarde solo con el día); para predecir un día completo se llama
dos veces, una por tramo. Ejemplo con una fecha fuera de todo el histórico actual:

In [ ]:
from src.feature_engineering import construir_features
df_pruebas = pd.DataFrame()
for fecha_prueba in ['2027-02-14', '2027-05-03', '2027-11-26']:
    for tramo_futuro in ['mañana', 'tarde']:
        fila = construir_features(fecha_prueba, tramo_futuro)
        df_pruebas = pd.concat([df_pruebas, fila], ignore_index=True, axis=1)
df_pruebas = df_pruebas.T.sort_values("fecha_cita")  # Transponer para que cada fila sea una fecha/tramo
print(df_pruebas.shape, df_pruebas)

# CUARTO PASO - Fase 6: Ingeniería de características (Feature Preprocessing)
En este notebook aplicamos las conclusiones obtenidas durante la fase de Exploración de Datos (EDA) sobre el dataset de reservas de **Oasis Spa Sevilla**. El objetivo es transformar el dataset bruto en una matriz de características puramente numérica y óptima para el entrenamiento de nuestros modelos de Machine Learning.

### Objetivos del preprocesamiento:
1. **Modelar el comportamiento temporal:** Agrupar variables de calendario para evitar sobreajuste y capturar patrones reales de demanda.
2. **Aplicar Reglas de Negocio:** Excluir datos ruidosos o no operativos (como los días de cierre del spa).
3. **Eliminar Redundancias:** Evitar la multicolinealidad eliminando variables correlacionadas o duplicadas.
4. **Automatización:** Validar la función centralizada `build_features` que se utilizará tanto en entrenamiento como en producción.

In [3]:
import sys
import os
import pandas as pd
import numpy as np
sys.path.append(os.path.abspath(os.path.join('..')))
from src.utils.preprocessing import build_features

In [ ]:
#Carga del dataset
ruta_train = '../data/processed/train_features.csv' 
ruta_test = '../data/processed/test_features.csv'
train_raw = pd.read_csv(ruta_train)
test_raw = pd.read_csv(ruta_test)

print("Datasets brutos cargados")
print(f"[Train, Filas iniciales: {train_raw.shape[0]} | Columnas: {len(train_raw.columns)}")
print(f"[Test,  Filas iniciales: {test_raw.shape[0]} | Columnas: {len(test_raw.columns)}")
print(f"\nColumnas iniciales en los datos: {train_raw.columns.tolist()}")

FileNotFoundError: [Errno 2] No such file or directory: '/data/processed/train_features.csv'

## Justificación de las transformaciones aplicadas

Para asegurar un modelo robusto y evitar que aprenda patrones sesgados, hemos aplicado las siguientes transformaciones matemáticas y de negocio en `src/preprocessing.py`:

### 1. Reglas de negocio y Llimpieza de filas
* **`es_cierre` (Exclusión de filas):** Los días en los que el Oasis Spa Sevilla permanece cerrado no aportan información sobre el comportamiento natural de la demanda (las citas son obligatoriamente 0). Entrenar con estos días sesgaría al modelo a la baja. Los excluimos del set de entrenamiento. En producción, una regla simple devolverá `0` citas en días de cierre sin llegar a consultar al modelo.

### 2. Ingeniería y reducción de redundancias (Columnas)
* **`tramo_tarde` en favor de `tramo`:** Transformamos la variable categórica `tramo` ("mañana"/"tarde") en una variable binaria donde `1` representa la tarde y `0` la mañana.
* **`grupo_dia` en favor de `nombre_dia` y `es_finde`:** El análisis del EDA demostró que la demanda se comporta en tres bloques claros (Lunes-Jueves, Viernes, y Fin de semana). Agruparlos reduce la cardinalidad de la variable y evita que el modelo sobreajuste para días específicos de la semana.
* **Señales débiles (`es_festivo` y `es_vispera_festivo`):** Aunque mostraron una correlación débil en el EDA, se mantienen inicialmente en el modelo para evaluar su impacto real en el entrenamiento. Serán las primeras candidatas a eliminar si decidimos simplificar el modelo (Principio de parsimonia).

### 3. Eliminación de variables temporales lineales
* Se eliminan las columnas originales de fecha (`fecha_cita`, `mes`, `anio`, `dia_semana`, `semana_iso`) para evitar que el modelo asuma relaciones lineales incorrectas con variables cíclicas.
* Se genera la variable **`dias_desde_inicio`** como nuestra variable de tendencia global para capturar el crecimiento del negocio en el tiempo de forma lineal.

In [ ]:
# 1.Transformar los datos de train y test aplicando build_features
X_train, y_train = build_features(train_raw)
X_test, y_test = build_features(test_raw)

print("Control de calidad de los datasets procesados")
print(f"Train, Filas resultantes (sin cierres): {X_train.shape[0]} | Columnas: {X_train.shape[1]}")
print(f"Test,  Filas resultantes (sin cierres): {X_test.shape[0]} | Columnas: {X_test.shape[1]}")

print(f"\n¿Quedan valores nulos en Train?: {X_train.isnull().sum().sum()}")
print(f"¿Quedan valores nulos en Test?:  {X_test.isnull().sum().sum()}")

#Comprobamos que todas las columnas son puramente numéricas en ambos
train_numerico = all(X_train.dtypes != 'object')
test_numerico = all(X_test.dtypes != 'object')

print(f"\n¿Son todas las columnas numéricas en Train?: {train_numerico}")
print(f"¿Son todas las columnas numéricas en Test?:  {test_numerico}")

#Comprobación de consistencia entre columnas
mismas_columnas = list(X_train.columns) == list(X_test.columns)
print(f"¿Tienen Train y Test exactamente las mismas columnas?: {mismas_columnas}")

## Control de calidad y tratamiento de datos (Missing Values & Drops)

Siguiendo las buenas prácticas metodológicas, evaluamos la necesidad de aplicar técnicas de imputación y selección automática de variables sobre nuestro conjunto de datos:

### 1. Gestión de valores nulos (Imputación)
En problemas de previsión de demanda con componente temporal, la imputación mediante medidas de tendencia central global (como la mediana o la media) puede distorsionar la estacionalidad diaria o semanal del negocio. 
* **Situación en Oasis Spa Sevilla:** Tras la fase de EDA, se ha verificado que el dataset no presenta registros nulos en las variables clave de calendario, tramos horarios o histórico de citas. 
* **Decisión:** Omitimos de forma intencionada el uso de herramientas como `SimpleImputer` para preservar la pureza temporal de la serie y evitar sesgar las predicciones futuras.

### 2. Selección de características y descarte de columnas (Feature Selection)
En lugar de aplicar un umbral genérico de descarte por porcentaje de nulos o varianza constante:
* Hemos realizado un filtrado guiado por **reglas de negocio** (exclusión de filas con `es_cierre == 1`).
* Descartamos variables redundantes analizadas en el EDA (`es_finde`, `tramo`, `nombre_dia`) directamente dentro de nuestro pipeline de preprocesamiento, sustituyéndolas por sus versiones optimizadas de menor cardinalidad (`grupo_dia`, `tramo_tarde`).

## Codificación de variables y escalado de características

### 1. Automatización y consistencia del Encoding (train & test)
Para evitar el riesgo de *Data Leakage* (fuga de datos) y garantizar la consistencia en el modelado, la codificación de variables categóricas (como los tramos horarios y las agrupaciones de días de la semana) se realiza de manera controlada y hermética dentro de nuestra función modular `build_features()`. 

Al aplicar esta misma función tanto a `train_raw` como a `test_raw`, garantizamos de forma idéntica:
* La eliminación de columnas excluidas o redundantes analizadas en el EDA (como `es_finde` o `nombre_dia`).
* Que el conjunto de entrenamiento y el de prueba adopten exactamente las mismas columnas numéricas y la misma estructura matemática final.

### 2. Prevención de Data Leakage en el escalado (feature scaling)
Dado que los algoritmos basados en árboles de decisión (como *Random Forest* o *XGBoost*) son invariantes a la escala, el escalado no es estrictamente necesario para los modelos de ensamble. Sin embargo, para poder compararlos de manera justa con modelos lineales (nuestro *Baseline*), aplicaremos una estandarización mediante `StandardScaler` a nuestra variable de tendencia continua (`dias_desde_inicio`).

Para simular un escenario real de producción y evaluar correctamente el modelo, seguimos una metodología estricta:
1. **Ajuste y transformación (`fit_transform`):** El objeto `StandardScaler` calcula la media ($\mu$) y la desviación estándar ($\sigma$) basándose **únicamente** en el conjunto de entrenamiento (`X_train`).
2. **Transformación pura (`transform`):** Aplicamos estos parámetros previamente calculados sobre el conjunto de test (`X_test`) sin recalcularlos. Esto impide que la información del conjunto de prueba "contamine" el entrenamiento del modelo. El escalador se exporta mediante `joblib` para poder usar exactamente la misma escala con datos nuevos en producción.

In [ ]:
import os
import pandas as pd
from sklearn.preprocessing import StandardScaler
import joblib


print("Aplicar build_features a train y test")
#Aplicar la función de preprocesamiento del script .py a ambos conjuntos por separado
X_train, y_train = build_features(train_raw)
X_test, y_test = build_features(test_raw)

print(f"Dimensiones de X_train: {X_train.shape}")
print(f"Dimensiones de X_test:  {X_test.shape}")

#Aplicar el escalado de forma segura
cols_a_escalar = ['dias_desde_inicio']

if all(col in X_train.columns for col in cols_a_escalar) and all(col in X_test.columns for col in cols_a_escalar):
    scaler = StandardScaler()
    
    #Ajustar y transformar únicamente con el conjunto de train
    X_train[cols_a_escalar] = scaler.fit_transform(X_train[cols_a_escalar])
    
    #Transformar el conjunto de test usando los parámetros de train
    X_test[cols_a_escalar] = scaler.transform(X_test[cols_a_escalar])
    
    #Creamos la carpeta de destino para el modelo si no existe
    os.makedirs('../src/models', exist_ok=True)
    
    #Guardamos el escalador ajustado para el entorno de producción
    joblib.dump(scaler, '../src/models/scaler.joblib')
    
    print("\nEscalado completado")
    print(f"Se ha guardado el escalador entrenado en: 'src/models/scaler.joblib'")
else:
    print("\n[Aviso] No se encontró la columna 'dias_desde_inicio' en alguno de los conjuntos.")

#Mostramos el resultado final de ambos conjuntos listos para modelar
print("\nVista previa del conjunto de entrenamiento procesado (X_train)")
display(X_train.head(2))

print("\nVista previa del conjunto de prueba procesado (X_test)")
display(X_test.head(2))

# QUINTO PASO - Modelado — comparativa, optimización y modelo final

Fase 5-6 de la guía (pasos 25-31) + persistencia (paso 36). Consume el pipeline
cerrado por las fases anteriores:

- `data/processed/train_features.csv` — dataset con las variables del EDA/FE
  (`feature_engineering.ipynb`).
- `src/preprocessing.build_features()` — filtra días de cierre, aplica one-hot a
  `grupo_dia`/`temporada`, recalcula la tendencia y elimina columnas redundantes
  (`feature_preprocessing.ipynb`).
- `src/models/scaler.joblib` — StandardScaler de `dias_desde_inicio`, ajustado
  sobre train por Preprocesado.

**El conjunto de test no se toca en este notebook.** Toda la selección de modelo
e hiperparámetros se hace con validación cruzada temporal sobre train; la
evaluación única contra test vive en `notebooks/evaluation.ipynb`.

In [ ]:
import sys, os
sys.path.append(os.path.abspath(os.path.join('..')))

import numpy as np
import pandas as pd
import joblib

from src.utils.preprocessing import build_features
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option('display.max_columns', None)
RANDOM_STATE = 42

## 1. Carga y preprocesado

Cargamos el dataset de features y aplicamos exactamente el mismo preprocesado
que definió el lane de Preprocesado. Guardamos aparte las fechas (que
`build_features` descarta) porque las necesitamos para los folds temporales y
para el baseline estacional.

In [ ]:
df = pd.read_csv('../data/processed/train_features.csv', parse_dates=['fecha_cita'])
df = df.sort_values(['fecha_cita', 'tramo']).reset_index(drop=True)

# mismo filtro de cierres que build_features, aplicado antes para conservar la fecha alineada
df_model = df[df['es_cierre'] != 1].reset_index(drop=True)

X, y = build_features(df_model)
fechas = df_model['fecha_cita']
tramos = df_model['tramo']

# escalado de la tendencia con el scaler ajustado por Preprocesado
scaler = joblib.load('../src/models/scaler.joblib')
X[['dias_desde_inicio']] = scaler.transform(X[['dias_desde_inicio']])

print(f"Filas: {X.shape[0]} (de {len(df)} tras excluir {len(df)-len(df_model)} tramos de cierre)")
print(f"Features ({X.shape[1]}): {list(X.columns)}")
print(f"Rango temporal: {fechas.min().date()} -> {fechas.max().date()}")

## 2. Métrica de evaluación (paso 25)

- **MAE (principal):** se lee directamente como "citas de más o de menos por
  tramo", la unidad con la que el negocio decide personal en sala.
- **RMSE (secundaria):** penaliza los errores grandes — infra-dotar un sábado
  por la tarde cuesta más que varios fallos pequeños.
- **R² (referencia):** cuánta varianza explica el modelo; sin unidades de negocio.
- **MAPE descartado:** el 21% de los tramos de mañana tienen 0 citas (EDA §2) y
  el error porcentual se dispara a infinito.

In [ ]:
def evaluar(y_true, y_pred, nombre=''):
    return {
        'modelo': nombre,
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'R2': r2_score(y_true, y_pred),
    }

tscv = TimeSeriesSplit(n_splits=5)
folds = list(tscv.split(X))
for i, (tr, va) in enumerate(folds, 1):
    print(f"Fold {i}: train {len(tr):4d} filas (hasta {fechas.iloc[tr].max().date()}) "
          f"-> valida {len(va):3d} filas ({fechas.iloc[va].min().date()} a {fechas.iloc[va].max().date()})")

## 3. Baselines por validación cruzada (paso 26)

Dos referencias mínimas, evaluadas **en los mismos folds temporales** que los
modelos para que la comparación sea justa:

1. **Media del fold de train**: ignora toda estructura.
2. **Estacional ingenuo (t-7)**: el valor real del mismo tramo 7 días naturales
   antes (vía join por fecha, robusto a huecos por cierres). Solo usa el pasado,
   así que no hay fuga temporal.

In [ ]:
# predicción t-7 por join fecha-7 días dentro del mismo tramo
hist = df_model[['fecha_cita', 'tramo', 'n_citas']].copy()
hist_shift = hist.rename(columns={'n_citas': 'pred_t7'})
hist_shift['fecha_cita'] = hist_shift['fecha_cita'] + pd.Timedelta(days=7)
t7 = hist.merge(hist_shift, on=['fecha_cita', 'tramo'], how='left')['pred_t7']

filas_cv = []
for tr, va in folds:
    y_tr, y_va = y.iloc[tr], y.iloc[va]
    filas_cv.append(evaluar(y_va, np.full(len(va), y_tr.mean()), 'Baseline media'))
    mask = t7.iloc[va].notna()
    filas_cv.append(evaluar(y_va[mask.values], t7.iloc[va][mask.values], 'Baseline estacional t-7'))

base_cv = pd.DataFrame(filas_cv).groupby('modelo').mean().round(2)
print(f"Cobertura t-7 en validación: {t7.notna().mean():.0%} de las filas")
base_cv

## 4. Comparativa de modelos (paso 27)

Seis algoritmos con parámetros por defecto, mismos folds. Lineales como
referencia interpretable; árboles y ensembles porque el EDA anticipó
interacciones no lineales (el efecto fin de semana se dispara en temporada
alta); KNN como no-paramétrico basado en distancias.

In [ ]:
modelos = {
    'LinearRegression': LinearRegression(),
    'Ridge': Ridge(random_state=RANDOM_STATE),
    'KNN': KNeighborsRegressor(),
    'DecisionTree': DecisionTreeRegressor(random_state=RANDOM_STATE),
    'RandomForest': RandomForestRegressor(random_state=RANDOM_STATE),
    'GradientBoosting': GradientBoostingRegressor(random_state=RANDOM_STATE),
}

filas = []
for nombre, modelo in modelos.items():
    for tr, va in folds:
        m = modelo.__class__(**modelo.get_params())
        m.fit(X.iloc[tr], y.iloc[tr])
        filas.append(evaluar(y.iloc[va], m.predict(X.iloc[va]), nombre))

comparativa = pd.DataFrame(filas).groupby('modelo').mean().sort_values('MAE').round(2)
comparativa

**Lectura:** la tabla anterior (media de los 5 folds temporales) decide qué
modelos pasan a optimización. Cualquier candidato debe batir a los dos
baselines — si no lo hace, no está aprendiendo nada que el calendario simple
no sepa ya.

## 5. Optimización de hiperparámetros (pasos 28-31)

Optimizamos los **dos mejores** de la comparativa (esperablemente los
ensembles). `RandomizedSearchCV` en lugar de búsqueda exhaustiva porque el
espacio combinado es grande, siempre con los mismos folds temporales
(`TimeSeriesSplit`) y optimizando MAE.

In [ ]:
top2 = comparativa.index[:2].tolist()
print('Modelos a optimizar:', top2)

espacios = {
    'RandomForest': (
        RandomForestRegressor(random_state=RANDOM_STATE),
        {
            'n_estimators': [200, 400, 600],
            'max_depth': [None, 6, 10, 16],
            'min_samples_leaf': [1, 2, 4, 8],
            'max_features': ['sqrt', 0.5, 1.0],
        },
    ),
    'GradientBoosting': (
        GradientBoostingRegressor(random_state=RANDOM_STATE),
        {
            'n_estimators': [150, 300, 500],
            'learning_rate': [0.03, 0.05, 0.1],
            'max_depth': [2, 3, 4],
            'min_samples_leaf': [1, 5, 10],
            'subsample': [0.8, 1.0],
        },
    ),
    'Ridge': (
        Ridge(random_state=RANDOM_STATE),
        {'alpha': [0.01, 0.1, 1.0, 10.0, 100.0]},
    ),
    'KNN': (
        KNeighborsRegressor(),
        {'n_neighbors': [3, 5, 9, 15, 25], 'weights': ['uniform', 'distance']},
    ),
    'DecisionTree': (
        DecisionTreeRegressor(random_state=RANDOM_STATE),
        {'max_depth': [3, 5, 8, 12, None], 'min_samples_leaf': [1, 5, 10, 20]},
    ),
    'LinearRegression': (LinearRegression(), {}),
}

busquedas = {}
for nombre in top2:
    est, grid = espacios[nombre]
    if not grid:
        continue
    rs = RandomizedSearchCV(
        est, grid, n_iter=25, cv=tscv, scoring='neg_mean_absolute_error',
        random_state=RANDOM_STATE, n_jobs=-1, refit=True,
    )
    rs.fit(X, y)
    busquedas[nombre] = rs
    print(f"\n{nombre}: mejor MAE CV = {-rs.best_score_:.3f}")
    print(f"  mejores parámetros: {rs.best_params_}")

In [ ]:
resumen = []
for nombre in top2:
    if nombre in busquedas:
        resumen.append({
            'modelo': nombre,
            'MAE CV (defecto)': comparativa.loc[nombre, 'MAE'],
            'MAE CV (optimizado)': round(-busquedas[nombre].best_score_, 2),
        })
tabla_opt = pd.DataFrame(resumen).set_index('modelo')
tabla_opt['mejora'] = (tabla_opt['MAE CV (defecto)'] - tabla_opt['MAE CV (optimizado)']).round(2)
tabla_opt

**Lectura:** documentamos el impacto de la optimización frente a los
parámetros por defecto (paso 31). El ganador se elige por MAE de CV — no por
test, que sigue intacto.

## 6. Modelo final y persistencia (paso 36)

`RandomizedSearchCV` con `refit=True` ya reentrena el mejor estimador sobre
**todo el train**. Guardamos un artefacto autocontenido con lo necesario para
inferencia y para la evaluación final:

- el modelo entrenado,
- la lista de columnas de X (para alinear el one-hot de test),
- metadatos de trazabilidad (métrica CV, fecha, features de origen).

El scaler de `dias_desde_inicio` ya está guardado por Preprocesado en
`src/models/scaler.joblib` — no lo duplicamos.

In [ ]:
ganador = min(busquedas, key=lambda n: -busquedas[n].best_score_)
modelo_final = busquedas[ganador].best_estimator_

artefacto = {
    'modelo': modelo_final,
    'nombre': ganador,
    'columnas': list(X.columns),
    'mae_cv': round(-busquedas[ganador].best_score_, 3),
    'entrenado_hasta': str(fechas.max().date()),
    'mejores_params': busquedas[ganador].best_params_,
}

os.makedirs('../src/models', exist_ok=True)
ruta = '../src/models/modelo_ocupacion.joblib'
joblib.dump(artefacto, ruta)
print(f"Guardado: {ruta}")
print(f"Modelo: {ganador} | MAE CV: {artefacto['mae_cv']} | entrenado hasta {artefacto['entrenado_hasta']}")
print("\nCómo cargarlo:")
print("  art = joblib.load('src/models/modelo_ocupacion.joblib')")
print("  modelo, columnas = art['modelo'], art['columnas']")

## 7. Qué queda para `evaluation.ipynb`

- Evaluación **única** contra test con este artefacto (paso 32).
- Análisis de residuos, real vs. predicho en el tiempo y por tramo (paso 33).
- Interpretabilidad: importancia de features del modelo final (paso 34).
- Contraste con el problema de negocio y limitaciones (paso 35).

# SEXTO PASO - Evaluación final contra test (pasos 32-35)

Primera —y única— vez que el conjunto de test entra en juego. Todo lo anterior
(EDA, features, selección de modelo, hiperparámetros) se decidió con validación
cruzada temporal sobre train; el resultado de este notebook es la estimación
honesta del rendimiento en producción.

Entradas:
- `src/models/modelo_ocupacion.joblib` — modelo final optimizado (RandomForest),
  entrenado sobre todo el train hasta 2026-01-24 (`notebooks/modeling.ipynb`).
- `src/models/scaler.joblib` — scaler de la tendencia (Preprocesado).
- `data/processed/test_features.csv` — 2026-01-25 → 2026-06-30, nunca usado antes.

In [ ]:
import sys, os
sys.path.append(os.path.abspath(os.path.join('..')))

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from src.utils.preprocessing import build_features
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option('display.max_columns', None)

def evaluar(y_true, y_pred, nombre=''):
    return {
        'modelo': nombre,
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'R2': r2_score(y_true, y_pred),
    }

## 1. Carga del modelo y preparación de test

Aplicamos a test **exactamente** el mismo pipeline que a train (filtro de
cierres, `build_features`, scaler) y alineamos las columnas del one-hot con las
que vio el modelo en entrenamiento — si a test le faltara alguna categoría
(p. ej. una temporada), se rellena a 0.

In [ ]:
art = joblib.load('../src/models/modelo_ocupacion.joblib')
modelo, columnas = art['modelo'], art['columnas']
print(f"Modelo: {art['nombre']} | MAE CV en train: {art['mae_cv']} | entrenado hasta {art['entrenado_hasta']}")

test = pd.read_csv('../data/processed/test_features.csv', parse_dates=['fecha_cita'])
test = test.sort_values(['fecha_cita', 'tramo']).reset_index(drop=True)
test_model = test[test['es_cierre'] != 1].reset_index(drop=True)
print(f"Test: {len(test)} filas, {len(test) - len(test_model)} tramos de cierre excluidos")
print(f"Periodo: {test_model['fecha_cita'].min().date()} -> {test_model['fecha_cita'].max().date()}")

X_test, y_test = build_features(test_model)
scaler = joblib.load('../src/models/scaler.joblib')
X_test[['dias_desde_inicio']] = scaler.transform(X_test[['dias_desde_inicio']])
X_test = X_test.reindex(columns=columnas, fill_value=0)

y_pred = modelo.predict(X_test)

## 2. Evaluación única contra test (paso 32)

Como contexto añadimos los dos baselines calculados **sobre el mismo periodo de
test**: la media de train y el estacional t-7 (que aquí puede usar el final de
train como historia — sigue siendo solo pasado, es una referencia legítima).

In [ ]:
train_hist = pd.read_csv('../data/processed/train_features.csv', parse_dates=['fecha_cita'])
train_hist = train_hist[train_hist['es_cierre'] != 1]

hist = pd.concat([
    train_hist[['fecha_cita', 'tramo', 'n_citas']],
    test_model[['fecha_cita', 'tramo', 'n_citas']],
], ignore_index=True)
hist_shift = hist.rename(columns={'n_citas': 'pred_t7'})
hist_shift['fecha_cita'] = hist_shift['fecha_cita'] + pd.Timedelta(days=7)
t7_test = test_model.merge(hist_shift, on=['fecha_cita', 'tramo'], how='left')['pred_t7']
mask_t7 = t7_test.notna()

resultados = pd.DataFrame([
    evaluar(y_test, np.full(len(y_test), train_hist['n_citas'].mean()), 'Baseline media de train'),
    evaluar(y_test[mask_t7.values], t7_test[mask_t7.values], 'Baseline estacional t-7'),
    evaluar(y_test, y_pred, f"{art['nombre']} (final)"),
]).set_index('modelo').round(2)
resultados

**Lectura clave:** contrastar el MAE de test con el estimado por CV
(1,74). Si están cerca, la validación temporal fue honesta y el modelo
generaliza; una desviación grande indicaría sobreajuste a train o un cambio de
régimen en el periodo de test.

## 3. Análisis de resultados (paso 33)

### 3.1 Real vs. predicho a lo largo del tiempo

In [ ]:
viz = test_model[['fecha_cita', 'tramo', 'n_citas']].copy()
viz['pred'] = y_pred

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
for ax, tr in zip(axes, ['mañana', 'tarde']):
    sub = viz[viz['tramo'] == tr]
    ax.plot(sub['fecha_cita'], sub['n_citas'], label='real', linewidth=1.2)
    ax.plot(sub['fecha_cita'], sub['pred'], label='predicho', linewidth=1.2, alpha=0.8)
    ax.set_title(f'Tramo {tr}')
    ax.set_ylabel('n_citas')
    ax.legend()
plt.tight_layout()
plt.show()

### 3.2 Dispersión y residuos

In [ ]:
residuos = y_test.values - y_pred

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].scatter(y_test, y_pred, alpha=0.4, s=18)
lims = [0, max(y_test.max(), y_pred.max()) + 1]
axes[0].plot(lims, lims, 'r--', linewidth=1)
axes[0].set_xlabel('Real'); axes[0].set_ylabel('Predicho'); axes[0].set_title('Real vs. predicho')

axes[1].scatter(y_pred, residuos, alpha=0.4, s=18)
axes[1].axhline(0, color='r', linestyle='--', linewidth=1)
axes[1].set_xlabel('Predicho'); axes[1].set_ylabel('Residuo'); axes[1].set_title('Residuos vs. predicho')

axes[2].hist(residuos, bins=25)
axes[2].set_xlabel('Residuo'); axes[2].set_title('Distribución de residuos')
plt.tight_layout()
plt.show()

print(f"Residuo medio (sesgo): {residuos.mean():+.2f} citas/tramo")
print(f"P90 del error absoluto: {np.percentile(np.abs(residuos), 90):.1f} citas")

### 3.3 ¿Dónde falla más? Error por segmento

El MAE global esconde asimetrías: lo desglosamos por los segmentos que el EDA
señaló como estructurales.

In [ ]:
seg = test_model[['tramo', 'grupo_dia', 'temporada', 'es_festivo']].copy()
seg['abs_err'] = np.abs(residuos)
seg['n_citas'] = y_test.values

for col in ['tramo', 'grupo_dia', 'temporada', 'es_festivo']:
    tabla = seg.groupby(col).agg(MAE=('abs_err', 'mean'), demanda_media=('n_citas', 'mean'), n=('abs_err', 'size')).round(2)
    print(f"--- por {col} ---")
    print(tabla.to_string(), '\n')

## 4. Interpretabilidad (paso 34)

Importancia de features del RandomForest final. Para un modelo de árboles es
importancia por reducción de impureza: qué variables usa más para partir.

In [ ]:
imp = pd.Series(modelo.feature_importances_, index=columnas).sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
imp.plot.barh(ax=ax)
ax.set_title('Importancia de features — modelo final')
ax.set_xlabel('Importancia (reducción de impureza)')
plt.tight_layout()
plt.show()

print(imp.sort_values(ascending=False).head(6).round(3).to_string())

## 5. Contraste con el problema de negocio (paso 35)

Para redactar tras ver los números de arriba, sobre tres ejes:

1. **¿Sirve para el cuadrante?** Un MAE de ~2 citas por tramo, sobre una demanda
   media de ~5 citas/tramo (y picos de 15+), significa que el modelo acota bien
   el orden de magnitud del tramo (flojo / normal / punta) aunque no la cita
   exacta. Para decidir personal por franjas eso es utilizable; para asignar
   cabina a cabina, no.
2. **¿Dónde equivocarse cuesta más?** Mirar el MAE de fin de semana y festivos
   (tabla §3.3): si el error se concentra en los tramos punta, el negocio debe
   tratar la predicción como suelo, no como techo, al dotar personal.
3. **Limitaciones honestas:** solo 2 ciclos anuales para validar estacionalidad;
   las últimas ~2 semanas de test están infra-contadas por la censura del export
   (lead time mediano 1 día); los cierres del negocio se excluyen — el modelo no
   predice si el spa abrirá, solo cuánta demanda tendrá si abre.

## 6. Conclusiones y acciones de mejora

- El modelo final bate a los dos baselines en test y su MAE es consistente con
  el estimado por CV — la metodología temporal fue honesta.
- **Acciones de mejora concretas** (ninguna es "probar otro modelo"):
  1. **Lags como features** (t-7, t-14) con protocolo de predicción rolling: el
     FE los descartó por definir antes el caso de uso; si el negocio confirma
     horizonte de predicción ≤7 días, son la mejora más prometedora (correlación
     ~0,55 ya medida).
  2. **Calendario oficial de cierres** del spa como dato de entrada, en vez de
     inferirlos de la demanda cero.
  3. **Más histórico**: reevaluar `es_fecha_comercial` y la estacionalidad anual
     cuando haya un tercer ciclo completo.
  4. **Predicción por intervalos** (cuantiles del bosque) para que el cuadrante
     trabaje con escenario pesimista/optimista, no con una cifra única.